In [1]:
!python --version

Python 3.11.9


In [2]:
DATASET_DIR = "../../datasets/"

import pathlib
OUTPUT_DIR = pathlib.Path(DATASET_DIR)
print(f"Dataset Directory: {OUTPUT_DIR}")

FILE_BPOM_DRUGS = "bpom_master_produk_komoditi_obat_20260907_12412.csv"
BPOM_DRUG_PATH = OUTPUT_DIR.joinpath(FILE_BPOM_DRUGS)
print(f"BPOM Drug Location: {BPOM_DRUG_PATH}")

FILE_BPOM_CLEANED = "bpom_drug_cleaned.csv"
BPOM_CLEANED_PATH = OUTPUT_DIR.joinpath(FILE_BPOM_CLEANED)
print(f"BPOM with cleaned version (no html tag, and clear sparation) will saved at Location: {BPOM_CLEANED_PATH}")

Dataset Directory: ..\..\datasets
BPOM Drug Location: ..\..\datasets\bpom_master_produk_komoditi_obat_20260907_12412.csv
BPOM with cleaned version (no html tag, and clear sparation) will saved at Location: ..\..\datasets\bpom_drug_cleaned.csv


In [3]:
import pandas as pd

drug_df = pd.read_csv(BPOM_DRUG_PATH)
print(drug_df.head())

   id             nama_produk              nie tanggal_terbit masa_berlaku  \
0   1                 MIRAVON  DTL0323406237A1     2025-07-02   2030-07-02   
1   2                ALUNBRIG  DKI2045600317A1     2020-04-01   2025-04-01   
2   3                 HYPERIL  DKL0305033810C1     2021-09-24   2026-09-12   
3   4                 EXAMEST  DKI2236200517A1     2022-11-01   2027-11-01   
4   5  AZITHROMYCIN DIHYDRATE  GKL2219621709A1     2022-01-11   2027-01-11   

                 bentuk_sediaan                                    kemasan  \
0                         SIRUP                       DUS, 1 BOTOL @ 60 ML   
1          TABLET SALUT SELAPUT   DUS, 2 BLISTER @ 14 TABLET SALUT SELAPUT   
2                  TABLET; 5 MG                  DUS, 10 STRIP @ 10 TABLET   
3   TABLET SALUT SELAPUT; 25 MG  DUS, 3 BLISTER @ 10 TABLET SALUT SELAPUT    
4  KAPLET SALUT SELAPUT; 530 MG      DUS, 1 STRIP @ 6 KAPLET SALUT SELAPUT   

                  komposisi                         pendaftar 

In [4]:
print(drug_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 23458 entries, 0 to 23457
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id                23458 non-null  int64
 1   nama_produk       23458 non-null  str  
 2   nie               23458 non-null  str  
 3   tanggal_terbit    23458 non-null  str  
 4   masa_berlaku      23458 non-null  str  
 5   bentuk_sediaan    23458 non-null  str  
 6   kemasan           23458 non-null  str  
 7   komposisi         19577 non-null  str  
 8   pendaftar         23451 non-null  str  
 9   nama_aplikasi     23458 non-null  str  
 10  diterbitkan_oleh  23458 non-null  str  
dtypes: int64(1), str(10)
memory usage: 2.0 MB
None


In [5]:
print(drug_df[['id','nama_produk','nie','komposisi']].head())

   id             nama_produk              nie                 komposisi
0   1                 MIRAVON  DTL0323406237A1  BROMHEXINE HYDROCHLORIDE
1   2                ALUNBRIG  DKI2045600317A1                       NaN
2   3                 HYPERIL  DKL0305033810C1                  RAMIPRIL
3   4                 EXAMEST  DKI2236200517A1                EXEMESTANE
4   5  AZITHROMYCIN DIHYDRATE  GKL2219621709A1    AZITHROMYCIN DIHYDRATE


In [6]:
drug_clean_df = drug_df.dropna(subset=['komposisi'])

In [7]:
print(drug_clean_df[['id','nama_produk','nie','komposisi']].head())

   id             nama_produk              nie                 komposisi
0   1                 MIRAVON  DTL0323406237A1  BROMHEXINE HYDROCHLORIDE
2   3                 HYPERIL  DKL0305033810C1                  RAMIPRIL
3   4                 EXAMEST  DKI2236200517A1                EXEMESTANE
4   5  AZITHROMYCIN DIHYDRATE  GKL2219621709A1    AZITHROMYCIN DIHYDRATE
5   6                 PASQUAM  DTL9822225029A1              DEXPANTHENOL


In [8]:
cleaned_compositions = []

In [9]:
def split_ingredients(compositions):
    ingredients = []
    if "<br>" in compositions:
        return [ composition.strip() for composition in compositions.split("<br>") ] 
    return [ composition.strip() for composition in compositions.split(".") ] 
    

In [10]:
from tqdm.auto import tqdm

results = []
for index, row in tqdm(
    drug_clean_df.iterrows(),
    total=len(drug_clean_df),
    desc="Processing drugs"
):
    results.append({
        'id': row['id'],
        'name': row['nama_produk'].strip(),
        'raw_ingredients': row['komposisi'],
        'ingredients': split_ingredients(row['komposisi']),
        'issued_date': row['tanggal_terbit'],
        'expired_date': row['masa_berlaku']
    })

Processing drugs:   0%|          | 0/19577 [00:00<?, ?it/s]

In [11]:
df = pd.DataFrame(results)
df = df.sort_values(by="issued_date")
cleaned_df = df.drop_duplicates(subset=["name", "raw_ingredients"], keep="last")

In [12]:
cleaned_df.to_csv(BPOM_CLEANED_PATH, index=False)
print(f"Cleaning data BPOM Successed, stored at {BPOM_CLEANED_PATH}")

Cleaning data BPOM Successed, stored at ..\..\datasets\bpom_drug_cleaned.csv
